# **RENAME FILE**

In [ ]:
# ======================================================================
#  COLAB-READY: rename_external_and_download.py
#  Description: Auto-detect ZIP, extract, rename images based on EXIF,
#               generate metadata CSV, and download the result as ZIP.
# ======================================================================
import zipfile
import shutil
import csv
import glob
from pathlib import Path
from PIL import Image
from google.colab import files

# ----------------------------------------------------------------------
#  CONFIGURATION - EDIT THIS SECTION
# ----------------------------------------------------------------------
SITE     = "Kediri"
CODE     = "KDR"

# Working & Output directories
WORK_DIR = Path("/content")                                  # where the zip file is located
EXTRACT_TO = Path("/content/kediri4")                        # extraction directory
OUT      = Path("/content/dataset_upload/Kediri")            # output directory ready for upload

DEFAULT_SESSION = "S1"          # used if photos are directly in the root (no session subfolder)
CAMERA   = "Sony a6000"
EXT      = {".jpg", ".jpeg", ".png"}
# ----------------------------------------------------------------------


# ======================================================================
#  STEP 0 - AUTO-DETECT ZIP (case-insensitive)
# ======================================================================
def find_zip(work_dir: Path, keyword: str = "kediri") -> Path:
    """Find a .zip file in work_dir containing the keyword (case-insensitive)."""
    zips = [Path(p) for p in glob.glob(str(work_dir / "*.zip"))]
    if not zips:
        raise FileNotFoundError(f"No .zip files found in {work_dir}")
    # Priority: name contains keyword
    match = [z for z in zips if keyword.lower() in z.name.lower()]
    chosen = match[0] if match else zips[0]
    print(f"[INFO] Detected Zip: {chosen}")
    return chosen


ZIP_PATH = find_zip(WORK_DIR, keyword="kediri")

# ======================================================================
#  STEP 1 - EXTRACT ZIP (skip if already extracted)
# ======================================================================
if not EXTRACT_TO.exists() or not any(EXTRACT_TO.iterdir()):
    print(f"[INFO] Extracting {ZIP_PATH} -> {EXTRACT_TO}")
    EXTRACT_TO.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_TO)
else:
    print(f"[INFO] {EXTRACT_TO} already exists, skipping extraction.")


# ======================================================================
#  STEP 2 - AUTO-DETECT RAW_SITE
#  Logic:
#   - If there's a session subfolder (S1, S2, ...) in root -> use root
#   - If there are images directly in root                 -> use root
#   - If there's only 1 wrapper subfolder                  -> go down 1 level
#   - If there are multiple subfolders (e.g., S1, S2)      -> use root
# ======================================================================
def is_session_folder(p: Path) -> bool:
    name = p.name.upper()
    return name.startswith("S") and name[1:].isdigit()


def detect_raw_site(root: Path) -> Path:
    if not root.exists():
        return root

    subs = [p for p in root.iterdir() if p.is_dir()]
    files = [p for p in root.iterdir() if p.is_file()]

    # 1) Found session subfolder (S1, S2, ...) -> root is RAW_SITE
    if any(is_session_folder(s) for s in subs):
        return root

    # 2) Found images directly in root -> root is RAW_SITE
    if any(f.suffix.lower() in EXT for f in files):
        return root

    # 3) Only 1 subfolder -> go down one level
    if len(subs) == 1:
        return subs[0]

    # 4) Fallback
    return root


RAW_SITE = detect_raw_site(EXTRACT_TO)
print(f"[INFO] RAW_SITE = {RAW_SITE}")


# ======================================================================
#  DISPLAY STRUCTURE (max 2 levels)
# ======================================================================
print("\n[EXTRACTED STRUCTURE]")
try:
    for p in sorted(EXTRACT_TO.rglob("*")):
        depth = len(p.relative_to(EXTRACT_TO).parts) - 1
        if depth <= 2:
            icon = "📁 " if p.is_dir() else "🖼️ "
            print("  " * depth + icon + p.name)
except Exception as e:
    print(f"[WARN] Failed to display structure: {e}")


# ======================================================================
#  HELPER FUNCTIONS
# ======================================================================
def exif_time(p: Path) -> str:
    try:
        ex = Image.open(p).getexif()
        dt = ex.get_ifd(0x8769).get(36867) or ex.get(306)   # DateTimeOriginal / DateTime
        return dt or ""
    except Exception:
        return ""


def list_images(folder: Path):
    return [f for f in folder.iterdir() if f.is_file() and f.suffix.lower() in EXT]


# ======================================================================
#  STEP 3 - RENAME + COPY + METADATA
# ======================================================================
if not RAW_SITE.exists():
    raise FileNotFoundError(
        f"Folder not found: {RAW_SITE}\n"
        f"Check the extracted structure above and adjust RAW_SITE."
    )

OUT.mkdir(parents=True, exist_ok=True)

# Sessions = subfolders (S1, S2, ...). If no subfolder -> DEFAULT_SESSION
sessions = sorted(p for p in RAW_SITE.iterdir() if p.is_dir())
if not sessions:
    sessions = [RAW_SITE]

rows = []
for sess_dir in sessions:
    sess = DEFAULT_SESSION if sess_dir == RAW_SITE else sess_dir.name
    files = list_images(sess_dir)
    times = {f: exif_time(f) for f in files}
    files.sort(key=lambda f: (times[f], f.name))

    for i, f in enumerate(files, 1):
        new = f"{CODE}_{sess}_{i:03d}{f.suffix.lower()}"
        dst = OUT / new
        if dst.exists():
            print(f"[SKIP] already exists, not overwritten: {new}")
            continue
        shutil.copy2(f, dst)
        dt = times[f]
        rows.append({
            "file": new,
            "original_file": f.name,
            "site": SITE,
            "session": sess,
            "date": dt[:10].replace(":", "-") if dt else "",
            "time": dt[11:16] if dt else "",
            "camera": CAMERA,
            "lcc_reading": "",
            "class": "",
            "notes": "",
        })

    no_exif = sum(1 for f in files if not times[f])
    print(f"[OK] {SITE}/{sess}: {len(files)} photos"
          + (f"  ({no_exif} without EXIF, sorted by filename)" if no_exif else ""))


# ======================================================================
#  STEP 4 - SAVE CSV
# ======================================================================
if rows:
    meta = OUT / f"external_metadata_{SITE}.csv"
    with open(meta, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print(f"\nFinished: {len(rows)} photos -> {OUT}")
    print(f"Metadata: {meta}")
else:
    print("No new photos were copied.")


# ======================================================================
#  STEP 5 - PREVIEW RESULT
# ======================================================================
print("\n[PREVIEW OF 10 RESULTING FILES]")
results = sorted(OUT.glob("*"))
for p in results[:10]:
    print("  -", p.name)
print(f"[TOTAL] {len([p for p in results if p.suffix.lower() in EXT])} images in {OUT}")


# ======================================================================
#  STEP 6 - DOWNLOAD RESULT: only the "Kediri" folder (renamed + CSV)
# ======================================================================
zip_target = Path("/content/hasil_kediri")       # zip name without .zip

if not OUT.exists():
    raise FileNotFoundError(f"Output folder does not exist: {OUT}")

# Zip folder OUT -> hasil_kediri.zip
print("\n[INFO] Creating ZIP archive...")
shutil.make_archive(str(zip_target), "zip", root_dir=OUT.parent, base_dir=OUT.name)

zip_file = zip_target.with_suffix(".zip")
print(f"[OK] Zip created: {zip_file}")
print(f"[SIZE] {zip_file.stat().st_size / 1e6:.2f} MB")

# Trigger download to local computer
print("[INFO] Starting download...")
files.download(str(zip_file))

# **EVALUATED EXTERNAL TEST SET**

In [ ]:
"""
eval_external_test.py
=====================
FINAL evaluation of the selected configuration on the external test set.

Perbaikan dibanding versi sebelumnya (revisi R1.10):
  - Zip external WAJIB persis; tidak lagi fallback ke zip lain.
  - Manifest MD5 zip + setiap citra/label dicatat (reproducibility).
  - Per-instance predictions disimpan (TP, FP, missed).
  - Confusion matrix: sel berisi COUNT dan % dari baris (konsisten dgn caption
    "counts summed across seeds"); kelas absen ditandai N/A; kolom "Missed".
  - Ekspektasi jumlah data disesuaikan (130/106/24/0).
  - Cek internal vs Table 13 (TABLE13_INTERNAL) benar-benar dipakai.
  - Docstring disamakan dengan konfigurasi aktual.
  - zip_md5 disimpan di settings.json.

External data:
    EXTERNAL_ZIP -> EXTERNAL_SRC
        New_Gsk/  -> Gresik, New_Kdr/ -> Kediri, New_Sda/ -> Sidoarjo

Internal data (Drive):
    MyDrive/dataset/Dataset_Roboflow     (raw)
    MyDrive/dataset/Dataset_Gamma_New    (processed)

Outputs (MyDrive/ablation_backup/external_eval/):
  - check_external_data.csv
  - external_manifest.csv                 (MD5 zip + file)
  - overall_metrics.md
  - perclass_ap.md
  - sitewise_tp_gt.md
  - confusion_external_raw_counts.csv
  - fig_confusion_external.png
  - correct_rate_bootstrap.csv            (combined + 3 site)
  - per_instance_predictions.csv          (per seed x instance)
  - settings.json                         (termasuk zip_md5)
  - per-seed CSVs

Protocol:
  - Mask IoU >= 0.50, Confidence >= 0.25, NMS IoU 0.45
  - Each object matched once
  - Missed GT -> "Missed" column; unmatched Prediction -> "False positive" row
  - Class order: 0=N23, 1=N34, 2=N45
"""
!pip install Ultralytics
import hashlib
import json, shutil, warnings, zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from ultralytics import YOLO

warnings.filterwarnings("ignore")

# ==========================================================================
#  CONFIGURATION
# ==========================================================================
DRIVE       = Path("/content/drive/MyDrive")
DATASET_DIR = DRIVE / "dataset"

EXTERNAL_ZIP    = Path("/content/New_Data_disamakan4.zip")
EXTERNAL_SRC    = Path("/content/tmp_external/source")
EXTERNAL_SITES  = ["New_Gsk", "New_Kdr", "New_Sda"]
SITE_LABEL_MAP  = {"New_Gsk": "Gresik", "New_Kdr": "Kediri", "New_Sda": "Sidoarjo"}

SITES: dict[str, Path] = {}
CLASS_REMAP = {"Gresik": {}, "Kediri": {}, "Sidoarjo": {}}
SITE_STATUS = {"Gresik":   "New sessions; site represented in main dataset",
               "Kediri":   "New sessions; site represented in main dataset",
               "Sidoarjo": "Site not represented in main dataset"}

RAW_DS      = DATASET_DIR / "Dataset_Roboflow"
PROC_DS     = DATASET_DIR / "Dataset_Gamma_New"
RUNS_ROOTS  = [Path("/content/runs/ablation"), DRIVE / "ablation_backup" / "runs"]
OUTPUT_DIR  = DRIVE / "ablation_backup" / "external_eval"
TMP         = Path("/content/tmp_external")

SELECTED = ("cross_raw_proc", "Raw → Processed", "raw", "processed", "processed")
SEEDS = [0, 1, 2]
TABLE13_INTERNAL = {"cross_raw_proc": (0.9773, 0.9524), "ft_raw_raw": (0.9781, 0.9490)}

# === FIX 6: ekspektasi jumlah data disesuaikan ===
EXP_TOT, EXP_N23, EXP_N34, EXP_N45 = 130, 106, 24, 0

CLASS_NAMES = ["N23", "N34", "N45"]
IMGSZ, VAL_BATCH = 640, 16
CONF_OP, NMS_IOU_OP, MATCH_IOU = 0.25, 0.45, 0.50
PREP = dict(gamma_min=0.7825, gamma_max=1.2026, gamma_target_mean=0.4835,
            edge_blur_sigma=0.6074, edge_strength=0.0469, hsv_saturation_gain=1.0171)
N_BOOT, BOOT_SEED = 2000, 42
FONT_PT, DPI = 11, 600
plt.rcParams.update({"font.family": "serif", "font.size": FONT_PT})
# ==========================================================================

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
K = len(CLASS_NAMES)
ZIP_MD5 = None   # diisi saat extract


# ---------------------------------------------------------------- utilities
def mount_drive():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"[INFO] Drive not mounted ({e})")


# === FIX 1 & 2: zip wajib persis + manifest MD5 ===
def extract_external_zip(zip_path: Path, dst: Path, sites: list[str]) -> dict[str, Path]:
    global ZIP_MD5
    if not zip_path.exists():
        raise FileNotFoundError(
            f"[FATAL] Zip external tidak ditemukan: {zip_path} "
            "(tidak memakai zip lain)"
        )
    if dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True, exist_ok=True)

    ZIP_MD5 = hashlib.md5(zip_path.read_bytes()).hexdigest()
    print(f"[ZIP] {zip_path.name} | MD5 {ZIP_MD5}")

    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dst)

    found: dict[str, Path] = {}
    for site in sites:
        cands = [c for c in dst.rglob(site) if c.is_dir()]
        if not cands:
            raise FileNotFoundError(
                f"[ZIP] Folder '{site}' tidak ditemukan di {zip_path.name}"
            )
        label = SITE_LABEL_MAP.get(site, site)
        found[label] = min(cands, key=lambda p: len(p.parts))
        print(f"[ZIP] {site:<8} -> {found[label]}")

    # manifest: MD5 setiap citra & label
    rows = []
    for label, root in found.items():
        for f in sorted(root.rglob("*")):
            if f.is_file() and f.suffix.lower() in IMG_EXT | {".txt"}:
                rows.append(dict(site=label,
                                 file=str(f.relative_to(root)),
                                 md5=hashlib.md5(f.read_bytes()).hexdigest()))
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(OUTPUT_DIR / "external_manifest.csv", index=False)
    print(f"[ZIP] Manifest: {len(rows)} file -> external_manifest.csv")
    return found


def find_split(ds_root, names=("test",)):
    for n in names:
        if (ds_root / n / "images").exists():
            return ds_root / n
    for p in ds_root.rglob("images"):
        if p.parent.name in names:
            return p.parent
    if (ds_root / "images").exists():
        return ds_root
    raise FileNotFoundError(f"split {names} not found in {ds_root}")


def resolve_weights(ds1, ds2, seed):
    run = f"stage1_{ds1}_s{seed}" if ds2 is None else f"stage2_{ds1}_to_{ds2}_s{seed}"
    for root in RUNS_ROOTS:
        w = root / run / "weights" / "best.pt"
        if w.exists():
            return w
    raise FileNotFoundError(f"best.pt {run} not found in {RUNS_ROOTS}")


def adaptive_preprocess(img, p=PREP):
    x = img.astype(np.float32) / 255.0
    g = np.clip(np.log(p["gamma_target_mean"]) / np.log(max(float(x.mean()), 1e-6)),
                p["gamma_min"], p["gamma_max"])
    x = np.clip(np.power(x, g) * 255, 0, 255).astype(np.uint8)
    bl = cv2.GaussianBlur(x, (0, 0), sigmaX=p["edge_blur_sigma"])
    x = np.clip(cv2.addWeighted(x, 1 + p["edge_strength"], bl, -p["edge_strength"], 0),
                0, 255).astype(np.uint8)
    h = cv2.cvtColor(x, cv2.COLOR_BGR2HSV).astype(np.float32)
    h[:, :, 1] = np.clip(h[:, :, 1] * p["hsv_saturation_gain"], 0, 255)
    return cv2.cvtColor(h.astype(np.uint8), cv2.COLOR_HSV2BGR)


# ---------------------------------------------------------------- 1. data check & prep
def check_and_prepare(site, src: Path, remap, domain):
    if (src / "test" / "images").exists() and (src / "test" / "labels").exists():
        src_img_dir = src / "test" / "images"
        src_lbl_dir = src / "test" / "labels"
    elif (src / "images").exists() and (src / "labels").exists():
        src_img_dir = src / "images"
        src_lbl_dir = src / "labels"
    else:
        raise FileNotFoundError(
            f"{site}: tidak ditemukan test/images+test/labels maupun images+labels di {src}"
        )

    dst = TMP / domain / site
    if dst.exists():
        shutil.rmtree(dst)
    (dst / "images").mkdir(parents=True)
    (dst / "labels").mkdir(parents=True)

    imgs = sorted(p for p in src_img_dir.iterdir() if p.suffix.lower() in IMG_EXT)
    labels = {p.stem for p in src_lbl_dir.glob("*.txt")}

    cnt = {n: 0 for n in CLASS_NAMES}
    no_label, bad = [], []
    for im in imgs:
        lp = src_lbl_dir / (im.stem + ".txt")
        lines_out = []
        if lp.exists():
            for line in lp.read_text().splitlines():
                v = line.split()
                if not v:
                    continue
                c = remap.get(int(v[0]), int(v[0]))
                if c not in range(K) or len(v) < 7 or (len(v) - 1) % 2:
                    bad.append(im.name)
                    continue
                cnt[CLASS_NAMES[c]] += 1
                lines_out.append(" ".join([str(c)] + v[1:]))
        else:
            no_label.append(im.name)
        (dst / "labels" / (im.stem + ".txt")).write_text(
            "\n".join(lines_out) + ("\n" if lines_out else "")
        )
        if domain == "processed":
            cv2.imwrite(str(dst / "images" / (im.stem + ".png")),
                        adaptive_preprocess(cv2.imread(str(im))))
        else:
            shutil.copy2(im, dst / "images" / im.name)

    orphan = sorted(labels - {p.stem for p in imgs})
    row = dict(site=site, images=len(imgs), **cnt, instances=sum(cnt.values()),
               images_without_label=len(no_label),
               invalid_label_lines=len(bad),
               labels_without_image=len(orphan))

    print(f"[DATA] {site:<9} {len(imgs):>3} images | "
          + " ".join(f"{n}={cnt[n]}" for n in CLASS_NAMES)
          + f" | total={row['instances']}")

    for msg, lst in [("without label", no_label),
                     ("invalid label lines", sorted(set(bad))),
                     ("labels without image", orphan)]:
        if lst:
            print(f"   [WARNING] {len(lst)} {msg}: {lst[:10]}{' ...' if len(lst) > 10 else ''}")
    absent = [n for n in CLASS_NAMES if cnt[n] == 0]
    if absent:
        print(f"   [INFO] classes absent in {site}: {absent} -> AP for these will be '–'")

    return dst, row


def write_yaml(name, dirs):
    TMP.mkdir(parents=True, exist_ok=True)
    y = TMP / f"{name}.yaml"
    imgs = [str(d / "images") for d in dirs]
    yaml.safe_dump({"path": "/", "train": imgs[0], "val": imgs[0],
                    "test": imgs if len(imgs) > 1 else imgs[0],
                    "names": {i: n for i, n in enumerate(CLASS_NAMES)}}, open(y, "w"))
    return y


# ---------------------------------------------------------------- 2. Ultralytics metrics
def val_metrics(model, yml):
    m = model.val(data=str(yml), split="test", imgsz=IMGSZ, batch=VAL_BATCH, plots=False,
                  verbose=False, project=str(TMP / "val"), name="run", exist_ok=True)
    s = m.seg
    idx = list(getattr(s, "ap_class_index", getattr(m, "ap_class_index", [])))
    per = {}
    for ci, n in enumerate(CLASS_NAMES):
        if ci in idx:
            j = idx.index(ci)
            per[n] = (float(s.ap50[j]), float(s.ap[j]))
        else:
            per[n] = (np.nan, np.nan)
    return dict(mAP50=float(s.map50), mAP50_95=float(s.map), P=float(s.mp), R=float(s.mr)), per


# ---------------------------------------------------------------- 3. confusion & TP/GT logic
def gt_masks(lp, H, W):
    cls, masks = [], []
    if lp.exists():
        for line in lp.read_text().splitlines():
            v = line.split()
            if len(v) < 7:
                continue
            pts = (np.array(v[1:], np.float32).reshape(-1, 2) * [W, H]).astype(np.int32)
            mk = np.zeros((H, W), np.uint8)
            cv2.fillPoly(mk, [pts], 1)
            cls.append(int(v[0]))
            masks.append(mk.astype(bool))
    return cls, masks


# === FIX 3: kembalikan juga list per-instance ===
def confusion_per_image(model, d):
    cm = np.zeros((K + 1, K + 1), int)
    per_img = []
    inst = []                                        # TP / FP / missed

    if (d / "images").exists():
        img_dir = d / "images"
        lbl_dir = d / "labels"
    else:
        img_dir = d / "test" / "images"
        lbl_dir = d / "test" / "labels"

    for p in sorted(f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXT):
        img = cv2.imread(str(p))
        H, W = img.shape[:2]
        gcls, gmask = gt_masks(lbl_dir / (p.stem + ".txt"), H, W)
        r = model.predict(img, imgsz=IMGSZ, conf=CONF_OP, iou=NMS_IOU_OP,
                          retina_masks=True, verbose=False)[0]
        if r.masks is not None:
            pm = r.masks.data.cpu().numpy() > 0.5
            pc = r.boxes.cls.cpu().numpy().astype(int)
            ps = r.boxes.conf.cpu().numpy()
        else:
            pm, pc, ps = np.zeros((0, H, W), bool), np.zeros(0, int), np.zeros(0)

        used, correct, fp = set(), 0, 0
        for i in np.argsort(-ps):
            best, bj = 0.0, -1
            for j, g in enumerate(gmask):
                if j in used:
                    continue
                iou = np.logical_and(pm[i], g).sum() / max(np.logical_or(pm[i], g).sum(), 1)
                if iou > best:
                    best, bj = iou, j
            if best >= MATCH_IOU:
                used.add(bj)
                cm[gcls[bj], pc[i]] += 1
                correct += int(gcls[bj] == pc[i])
                inst.append(dict(site=d.name, filename=p.name,
                                 image=f"{d.name}/{p.name}",
                                 gt=CLASS_NAMES[gcls[bj]],
                                 pred=CLASS_NAMES[pc[i]],
                                 conf=float(ps[i]), mask_iou=float(best)))
            else:
                cm[K, pc[i]] += 1
                fp += 1
                inst.append(dict(site=d.name, filename=p.name,
                                 image=f"{d.name}/{p.name}",
                                 gt="FP",
                                 pred=CLASS_NAMES[pc[i]],
                                 conf=float(ps[i]), mask_iou=float(best)))

        for j in range(len(gmask)):
            if j not in used:
                cm[gcls[j], K] += 1
                inst.append(dict(site=d.name, filename=p.name,
                                 image=f"{d.name}/{p.name}",
                                 gt=CLASS_NAMES[gcls[j]],
                                 pred="missed",
                                 conf=np.nan, mask_iou=np.nan))

        per_img.append(dict(site=d.name, filename=p.name,
                            image=f"{d.name}/{p.name}", n_gt=len(gmask),
                            n_correct=correct, n_fp=fp))

    return cm, per_img, inst


# === FIX 5: confusion matrix konsisten dengan caption ===
def plot_cm(counts, out_png):
    """Plot confusion matrix: counts + % dari baris. Baris kelas absen = N/A.
    Kolom 'Missed' untuk GT tak terdeteksi; baris 'False positive' untuk
    prediksi tanpa GT. Judul di caption naskah."""
    present = [i for i in range(K) if counts[i].sum() > 0]
    rows = present + [K]                 # baris FP
    cols = list(range(K)) + [K]          # kolom missed
    sub = counts[np.ix_(rows, cols)]
    rs = sub.sum(1, keepdims=True)
    pct = np.divide(sub, rs, out=np.zeros_like(sub, float), where=rs > 0) * 100
    ylab = [CLASS_NAMES[i] for i in present] + ["False positive"]
    xlab = CLASS_NAMES + ["Missed"]

    fig, ax = plt.subplots(figsize=(4.5, 3.6), dpi=DPI)
    ax.imshow(pct, cmap=plt.cm.Blues, vmin=0, vmax=100)
    for i in range(len(rows)):
        for j in range(len(cols)):
            if i == len(rows) - 1 and j == len(cols) - 1:
                continue
            text = f"{sub[i, j]}\n({pct[i, j]:.0f}%)"
            ax.text(j, i, text, ha="center", va="center", fontsize=FONT_PT,
                    color="white" if pct[i, j] > 50 else "black")
    ax.set_xticks(range(len(cols)), xlab, rotation=45, ha="right")
    ax.set_yticks(range(len(rows)), ylab)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
    plt.tight_layout(); fig.savefig(out_png, bbox_inches="tight"); plt.close(fig)


def bootstrap_rate(per_img, n_boot=N_BOOT, seed=BOOT_SEED):
    df = pd.DataFrame(per_img)
    g = df.groupby("image")[["n_gt", "n_correct"]].sum()
    gt, ok = g.n_gt.values, g.n_correct.values
    if gt.sum() == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(gt), (n_boot, len(gt)))
    rates = ok[idx].sum(1) / np.maximum(gt[idx].sum(1), 1)
    return ok.sum() / gt.sum(), np.percentile(rates, 2.5), np.percentile(rates, 97.5)


# ---------------------------------------------------------------- main
def main():
    mount_drive()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    global SITES
    SITES = extract_external_zip(EXTERNAL_ZIP, EXTERNAL_SRC, EXTERNAL_SITES)
    missing = [s for s in SITE_LABEL_MAP.values() if s not in SITES]
    if missing:
        raise SystemExit(f"[FATAL] Site berikut tidak ditemukan di zip: {missing}")

    cid, label, d1, d2, domain = SELECTED
    weights = {s: resolve_weights(d1, d2, s) for s in SEEDS}
    names = [YOLO(str(weights[SEEDS[0]]), task="segment").names[i] for i in range(K)]
    assert names == CLASS_NAMES, f"Model class order {names} != {CLASS_NAMES}"
    print(f"[OK] Selected configuration: {label} | input {domain} | seeds {SEEDS}")

    # --- Data Check & Prep ---
    site_dirs, rows = {}, []
    for site, src in SITES.items():
        if not src.exists():
            print(f"[SKIP] {site}: folder {src} not found")
            continue
        site_dirs[site], row = check_and_prepare(site, src, CLASS_REMAP.get(site, {}), domain)
        rows.append(row)

    if not site_dirs:
        raise SystemExit("No external test folders found.")

    chk = pd.DataFrame(rows)
    tot = chk.drop(columns="site").sum()
    tot["site"] = "External (combined)"
    chk = pd.concat([chk, tot.to_frame().T], ignore_index=True)
    chk.to_csv(OUTPUT_DIR / "check_external_data.csv", index=False)

    print("\n[AUTOMATIC DATA VALIDATION]")
    act_tot = int(tot["instances"]); act_n23 = int(tot["N23"])
    act_n34 = int(tot["N34"]);       act_n45 = int(tot["N45"])
    if (act_tot, act_n23, act_n34, act_n45) != (EXP_TOT, EXP_N23, EXP_N34, EXP_N45):
        print("   [WARNING] Distribusi external test berbeda dari ekspektasi.")
        print(f"   Expected: Tot={EXP_TOT}, N23={EXP_N23}, N34={EXP_N34}, N45={EXP_N45}")
        print(f"   Actual  : Tot={act_tot}, N23={act_n23}, N34={act_n34}, N45={act_n45}")
        print("   Lanjut dengan data aktual...")
    else:
        print("   [OK] Distribusi external test sesuai ekspektasi.")

    sets = {"Internal test": [find_split(PROC_DS if domain == "processed" else RAW_DS)],
            "External (combined)": list(site_dirs.values())}
    if len(site_dirs) > 1:
        sets.update({f"External – {s}": [d] for s, d in site_dirs.items()})

    int_dir = sets["Internal test"][0]

    int_cnt = {c: 0 for c in CLASS_NAMES}
    int_img_dir = int_dir / "images" if (int_dir / "images").exists() else int_dir / "test" / "images"
    int_lbl_dir = int_dir / "labels" if (int_dir / "labels").exists() else int_dir / "test" / "labels"
    int_cnt["images"] = len(list(int_img_dir.iterdir()))
    for lp in int_lbl_dir.glob("*.txt"):
        for line in lp.read_text().splitlines():
            if line.strip():
                int_cnt[CLASS_NAMES[int(line.split()[0])]] += 1

    ymls = {k: write_yaml(f"set{i}", v) for i, (k, v) in enumerate(sets.items())}

    # --- Evaluate Per Seed ---
    ov, pc, tpgt = [], [], []
    per_img_external = []
    per_img_by_set   = {}
    cms_3seeds = []
    inst_rows = []                                    # === FIX 3 ===

    for s in SEEDS:
        model = YOLO(str(weights[s]), task="segment")
        for name, dirs in sets.items():
            m, per = val_metrics(model, ymls[name])
            ov.append(dict(set=name, seed=s, **m))
            for c, (a50, a5095) in per.items():
                pc.append(dict(set=name, seed=s, cls=c, AP50=a50, AP50_95=a5095))

            if name == "Internal test":
                continue

            cm_s = np.zeros((K + 1, K + 1), int)
            for d in dirs:
                cm_d, pi_d, inst_d = confusion_per_image(model, d)
                cm_s += cm_d
                if name.startswith("External"):
                    per_img_by_set.setdefault(name, []).extend([dict(seed=s, **r) for r in pi_d])
                if name == "External (combined)":
                    per_img_external.extend([dict(seed=s, **r) for r in pi_d])
                    inst_rows += [dict(seed=s, **r) for r in inst_d]

            if name == "External (combined)":
                cms_3seeds.append(cm_s)

            for ci, c in enumerate(CLASS_NAMES):
                tpgt.append(dict(set=name, seed=s, cls=c,
                                 TP=int(cm_s[ci, ci]), GT=int(cm_s[ci].sum())))
            tpgt.append(dict(set=name, seed=s, cls="FP",
                             TP=int(cm_s[K, :K].sum()), GT=np.nan))
        print(f"[OK] seed {s} completed")

    ov, pc, tpgt = pd.DataFrame(ov), pd.DataFrame(pc), pd.DataFrame(tpgt)
    ov.to_csv(OUTPUT_DIR / "overall_metrics_per_seed.csv", index=False)
    pc.to_csv(OUTPUT_DIR / "perclass_ap_per_seed.csv", index=False)
    tpgt.to_csv(OUTPUT_DIR / "tp_gt_per_seed.csv", index=False)

    # === Simpan prediksi per objek + CSV khusus audit kesalahan per foto ===
    df_inst = pd.DataFrame(inst_rows)
    df_inst.to_csv(OUTPUT_DIR / "per_instance_predictions.csv", index=False)
    print(f"[OK] per_instance_predictions.csv: {len(df_inst)} baris")

    # Detail semua kesalahan per objek: salah kelas, missed, dan false positive.
    if not df_inst.empty:
        is_fp = df_inst["gt"].eq("FP")
        is_missed = df_inst["pred"].eq("missed")
        is_wrong_class = (~is_fp) & (~is_missed) & df_inst["gt"].ne(df_inst["pred"])
        df_errors = df_inst[is_fp | is_missed | is_wrong_class].copy()
        df_errors["error_type"] = np.select(
            [is_fp.loc[df_errors.index], is_missed.loc[df_errors.index],
             is_wrong_class.loc[df_errors.index]],
            ["false_positive", "missed", "wrong_class"],
            default="unknown"
        )
        df_errors.to_csv(OUTPUT_DIR / "per_instance_errors.csv", index=False)

        # Ringkasan per foto, menggabungkan hasil ketiga seed.
        # Angka *_total adalah jumlah kejadian lintas seed, bukan jumlah foto unik.
        summary_rows = []
        for (site, filename, image), g in df_inst.groupby(
                ["site", "filename", "image"], dropna=False, sort=True):
            fp_mask = g["gt"].eq("FP")
            missed_mask = g["pred"].eq("missed")
            matched_mask = (~fp_mask) & (~missed_mask)
            wrong_mask = matched_mask & g["gt"].ne(g["pred"])
            n34_gt_mask = g["gt"].eq("N34")
            n34_wrong_mask = n34_gt_mask & wrong_mask
            n34_missed_mask = n34_gt_mask & missed_mask

            # Jumlah seed yang mengalami minimal satu kesalahan N34 pada foto ini.
            n34_error_rows = g[n34_wrong_mask | n34_missed_mask]
            seeds_with_n34_error = int(n34_error_rows["seed"].nunique())

            # Setiap GT muncul sekali per seed (baik match maupun missed).
            gt_rows = g[~fp_mask]
            gt_count_total = int(len(gt_rows))
            n_correct_total = int((matched_mask & g["gt"].eq(g["pred"])).sum())
            wrong_total = int(wrong_mask.sum())
            missed_total = int(missed_mask.sum())
            fp_total = int(fp_mask.sum())
            n34_gt_total = int(n34_gt_mask.sum())

            summary_rows.append({
                "site": site,
                "filename": filename,
                "image": image,
                "seeds_evaluated": int(g["seed"].nunique()),
                "GT_instances_across_seeds": gt_count_total,
                "correct_instances_across_seeds": n_correct_total,
                "wrong_class_total": wrong_total,
                "missed_total": missed_total,
                "false_positive_total": fp_total,
                "all_errors_total": wrong_total + missed_total + fp_total,
                "N34_GT_across_seeds": n34_gt_total,
                "N34_wrong_class_total": int(n34_wrong_mask.sum()),
                "N34_missed_total": int(n34_missed_mask.sum()),
                "N34_errors_total": int(n34_wrong_mask.sum() + n34_missed_mask.sum()),
                "seeds_with_N34_error": seeds_with_n34_error,
                "N34_wrong_class_predictions": "; ".join(
                    f"{int(r.seed)}:{r.pred}" for r in g[n34_wrong_mask].itertuples()
                )
            })

        df_summary = pd.DataFrame(summary_rows)
        df_summary = df_summary.sort_values(
            ["N34_errors_total", "all_errors_total", "wrong_class_total", "missed_total"],
            ascending=[False, False, False, False]
        )
        df_summary.to_csv(OUTPUT_DIR / "per_image_error_summary.csv", index=False)

        # Ringkasan satu baris per foto dan seed, untuk memeriksa seed mana yang keliru.
        seed_rows = []
        for (site, filename, image, seed), g in df_inst.groupby(
                ["site", "filename", "image", "seed"], dropna=False, sort=True):
            fp_mask = g["gt"].eq("FP")
            missed_mask = g["pred"].eq("missed")
            matched_mask = (~fp_mask) & (~missed_mask)
            wrong_mask = matched_mask & g["gt"].ne(g["pred"])
            n34_gt_mask = g["gt"].eq("N34")
            n34_wrong_mask = n34_gt_mask & wrong_mask
            n34_missed_mask = n34_gt_mask & missed_mask
            seed_rows.append({
                "site": site, "filename": filename, "image": image, "seed": int(seed),
                "GT_instances": int((~fp_mask).sum()),
                "correct_instances": int((matched_mask & g["gt"].eq(g["pred"])).sum()),
                "wrong_class": int(wrong_mask.sum()),
                "missed": int(missed_mask.sum()),
                "false_positive": int(fp_mask.sum()),
                "N34_GT": int(n34_gt_mask.sum()),
                "N34_wrong_class": int(n34_wrong_mask.sum()),
                "N34_missed": int(n34_missed_mask.sum()),
                "N34_errors": int(n34_wrong_mask.sum() + n34_missed_mask.sum())
            })
        pd.DataFrame(seed_rows).sort_values(
            ["N34_errors", "wrong_class", "missed"], ascending=False
        ).to_csv(OUTPUT_DIR / "per_image_error_by_seed.csv", index=False)

        print(f"[OK] per_instance_errors.csv: {len(df_errors)} baris kesalahan")
        print(f"[OK] per_image_error_summary.csv: {len(df_summary)} foto; "
              "diurutkan dari kesalahan N34 terbanyak")
        print("[OK] per_image_error_by_seed.csv: ringkasan setiap foto per seed")
    else:
        print("[WARNING] Tidak ada baris prediksi; CSV audit kesalahan tidak dibuat.")

    fmt = lambda v: "–" if np.all(np.isnan(v)) else f"{np.nanmean(v):.4f} ± {np.nanstd(v, ddof=1):.4f}"

    # --- (2) Overall Table (Table 16) ---
    order = list(sets.keys())
    L = [f"Table 16. Performance of the selected {label} configuration on the internal test subset and the "
         "combined and site-specific external test sets\n",
         "| Evaluation set | Images | Instances (N23/N34/N45) | mAP@0.5 | mAP@0.5:0.95 | P | R |",
         "|---|---|---|---|---|---|---|"]
    cnt_map = {r.site: r for r in chk.itertuples()}
    for name in order:
        g = ov[ov.set == name]
        if name == "Internal test":
            info = (f"{int_cnt['images']} | {sum(int_cnt[c] for c in CLASS_NAMES)} ("
                    + "/".join(str(int_cnt[c]) for c in CLASS_NAMES) + ")")
        else:
            key = "External (combined)" if name == "External (combined)" else name.replace("External – ", "")
            r = cnt_map[key]
            info = f"{r.images} | {r.instances} ({r.N23}/{r.N34}/{r.N45})"
        shown = {"Internal test": "Internal test subset",
                 "External (combined)": "External test set (combined)"}.get(
                 name, name.replace("External – ", "External – "))
        L.append(f"| {shown} | {info} | "
                 + " | ".join(fmt(g[c].values) for c in ["mAP50", "mAP50_95", "P", "R"]) + " |")
    L.append("\nNote: Results are reported as mean ± SD across three seed-specific models. The combined external "
             "test set provides the primary final performance estimate. mAP is averaged over the classes present in each set.")
    (OUTPUT_DIR / "overall_metrics.md").write_text("\n".join(L), encoding="utf-8")
    print("\n" + "\n".join(L))

    # === FIX 7: cek internal vs Table 13 ===
    ref = TABLE13_INTERNAL.get(cid)
    gi = ov[ov.set == "Internal test"]
    if ref:
        d50 = abs(gi.mAP50.mean() - ref[0])
        d95 = abs(gi.mAP50_95.mean() - ref[1])
        status = "OK" if max(d50, d95) < 0.002 else "⚠️ BERBEDA — cek weights/data"
        print(f"[CHECK] Internal vs Table 13: ΔmAP50={d50:.4f}, ΔmAP50-95={d95:.4f}  {status}")

    # --- (3) Per-class AP ---
    ext_present = [c for c in CLASS_NAMES if getattr(cnt_map["External (combined)"], c) > 0]
    L = ["Table [X]. Per-class mask AP of the selected configuration on the internal and external test sets "
         "(mean ± SD over three seeds)\n",
         "| Class | Instances (internal / external) | mAP@0.5 internal | mAP@0.5:0.95 internal | "
         "mAP@0.5 external | mAP@0.5:0.95 external |", "|---|---|---|---|---|---|"]
    inst_int = int_cnt
    for c in CLASS_NAMES:
        gi_ = pc[(pc.set == "Internal test") & (pc.cls == c)]
        ge_ = pc[(pc.set == "External (combined)") & (pc.cls == c)]
        L.append(f"| {c} | {inst_int[c]} / {getattr(cnt_map['External (combined)'], c)} | "
                 f"{fmt(gi_.AP50.values)} | {fmt(gi_.AP50_95.values)} | "
                 f"{fmt(ge_.AP50.values)} | {fmt(ge_.AP50_95.values)} |")
    gi_, ge_ = ov[ov.set == "Internal test"], ov[ov.set == "External (combined)"]
    L.append(f"| All | {sum(int_cnt[c] for c in CLASS_NAMES)} / "
             f"{getattr(cnt_map['External (combined)'], 'instances')} | "
             f"{fmt(gi_.mAP50.values)} | {fmt(gi_.mAP50_95.values)} | "
             f"{fmt(ge_.mAP50.values)} | {fmt(ge_.mAP50_95.values)} |")
    sub = pc[(pc.set == "Internal test") & (pc.cls.isin(ext_present))].groupby("seed")[["AP50", "AP50_95"]].mean()
    L.append(f"\nInternal-test mAP recomputed over the classes present in the external set ({', '.join(ext_present)}): "
             f"mAP@0.5 = {fmt(sub.AP50.values)}, mAP@0.5:0.95 = {fmt(sub.AP50_95.values)}")
    L.append("'–' = class absent from the set.")
    (OUTPUT_DIR / "perclass_ap.md").write_text("\n".join(L), encoding="utf-8")
    print("\n" + "\n".join(L))

    # --- (4) Site-wise TP/GT per seed (Table 17) ---
    L = ["Table 17. Site-wise descriptive results on the external test set\n",
         "| Site | Status | Images | "
         + " | ".join(f"{c} TP/GT (seeds {', '.join(map(str, SEEDS))})" for c in CLASS_NAMES)
         + f" | False positives (seeds {', '.join(map(str, SEEDS))}) |",
         "|---|---|---|" + "---|" * (K + 1)]
    site_rows = [n for n in order if n.startswith("External – ")] or ["External (combined)"]
    for name in site_rows:
        key = name.replace("External – ", "") if name.startswith("External – ") else list(site_dirs)[0]
        cells = []
        for c in CLASS_NAMES:
            g = tpgt[(tpgt.set == name) & (tpgt.cls == c)].sort_values("seed")
            cells.append("–" if g.GT.sum() == 0 else "; ".join(f"{int(a)}/{int(b)}"
                                                               for a, b in zip(g.TP, g.GT)))
        fp = tpgt[(tpgt.set == name) & (tpgt.cls == "FP")].sort_values("seed").TP
        L.append(f"| {key} | {SITE_STATUS.get(key, '[status]')} | {cnt_map[key].images} | "
                 + " | ".join(cells) + " | " + "; ".join(map(str, fp)) + " |")
    L.append(f"\nNote: TP = correctly segmented and classified instances (mask IoU ≥ {MATCH_IOU}, "
             f"conf = {CONF_OP}, NMS IoU = {NMS_IOU_OP}); GT = annotated instances; '–' = class absent at the site.")
    (OUTPUT_DIR / "sitewise_tp_gt.md").write_text("\n".join(L), encoding="utf-8")
    print("\n" + "\n".join(L))

    # --- (5) Confusion Matrix & (6) Bootstrap per-set ---
    boots = []
    for name in order:
        if name == "Internal test":
            continue

        if name == "External (combined)":
            actual_counts_sum = np.zeros((K + 1, K + 1), int)
            for cm in cms_3seeds:
                actual_counts_sum += cm

            df_cm = pd.DataFrame(actual_counts_sum, index=CLASS_NAMES + ["False positive"],
                                 columns=CLASS_NAMES + ["Missed"])
            df_cm.to_csv(OUTPUT_DIR / "confusion_external_raw_counts.csv")
            plot_cm(actual_counts_sum, OUTPUT_DIR / "fig_confusion_external.png")

        rate, lo, hi = bootstrap_rate(per_img_by_set[name])
        boots.append(dict(set=name, correct_rate=rate, ci95_low=lo, ci95_high=hi,
                          n_images=len({r['image'] for r in per_img_by_set[name]})))

    bt = pd.DataFrame(boots)
    bt.to_csv(OUTPUT_DIR / "correct_rate_bootstrap.csv", index=False)
    print("\nProportion of correctly classified instances + 95% bootstrap CI per image:")
    print(bt.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

    # === FIX 8: settings.json menyertakan zip_md5 ===
    json.dump(dict(selected=label, seeds=SEEDS, conf=CONF_OP, nms_iou=NMS_IOU_OP,
                   match_iou=MATCH_IOU,
                   external_zip=str(EXTERNAL_ZIP),
                   zip_md5=ZIP_MD5,
                   external_src=str(EXTERNAL_SRC),
                   sites={k: str(v) for k, v in SITES.items()},
                   class_remap=CLASS_REMAP),
              open(OUTPUT_DIR / "settings.json", "w"), indent=2, ensure_ascii=False)
    print(f"\n[DONE] All outputs saved in {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [Ultralytics]
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Mounted at /content/drive
[ZIP] New_Data_disamakan4.zip | MD5 80805ea84abd901136c7ea89f37758aa
[ZIP] New_Gsk  -> /content/tmp_external/source/NEW DATA_2/New_Gsk
[ZIP] New_Kdr  -> /content/tmp_external/source/NEW DATA_2/New_Kdr
[ZIP] New_Sda  -> /content/tmp_external/source/NEW DATA_2/New_Sda
[ZIP] Manifest: 266 file -> external_manifest.csv
[OK] Selected configuration: Raw → Processed | input processed | seeds [0, 1, 2]
[DATA] Gresik     45 images | N23=43 N34=2 N45=0 | total=45
   [INFO] classes absent in Gresik: ['N45'] -> AP for these will be '–'
[DATA] Kediri  